# Description

The purpose of this notebook is to push our dataset to Hugging Face in a easy to setup format and use so that we can easily change models and processors and datasets and validate everything easily. Otherwise, adapting such bits to our cases is a nightmare

# Imports

In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv


In [2]:
load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')

In [3]:
! hf repos tag create jddomingues/brexpert-mm v2 --repo-type dataset --revision main -m "Dataset v2"

✓ Tag created
  tag: v2
  repo_type: dataset
  repo_id: jddomingues/brexpert-mm


In [ ]:
# The dataset preview is built locally below; no remote dataset is modified.

In [ ]:
# Run the local build cell before inspecting images.

# Load the current VQA dataset

In [ ]:
TRAIN_JSONL_PATH = "../data/robust_vqa_split/vqa_train.jsonl"
VAL_JSONL_PATH = "../data/robust_vqa_split/vqa_val.jsonl"
TEST_JSONL_PATH = "../data/robust_vqa_split/vqa_test.jsonl"

TRAIN_CSV_PATH = "../data/report_generation_split/all-rg-train.csv"
TEST_CSV_PATH = "../data/report_generation_split/all-rg-test.csv"
VAL_CSV_PATH = "../data/report_generation_split/all-rg-val.csv"

In [ ]:
train_vqa = pd.read_json(TRAIN_JSONL_PATH, lines=True)
train_csv = pd.read_csv(TRAIN_CSV_PATH)
train_csv["_memmap_idx"] = np.arange(len(train_csv), dtype=np.int64)

In [ ]:
merge_cols = [
    "id",
    "_memmap_idx",
    "birads",
    "collapsed_birads",
    "assessment_scope",
    "exam_id",
    "breast_id",
    "segmentation",
    "modality",
    "dataset",
    "patient",
]
out = train_vqa.merge(
    train_csv[merge_cols],
    left_on="source_id",
    right_on="id",
    how="left",
    suffixes=("_vqa", "_rg"),
)

In [ ]:
out.head(2)

In [ ]:
out.iloc[0]["conversation"]

# Count Max Token Length Response

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
            "microsoft/biogpt",
            trust_remote_code=True,
        )

In [ ]:
min_length = 100
max_length = -1
all = []

# for row in tqdm(out.itertuples(), total=len(out)):
#     for round in row.conversation:
#         if round["role"] == "assistant":
#             toks = tokenizer(round["text"], add_special_tokens=False)["input_ids"]
#             if len(toks) < min_length:
#                 min_length = len(toks)
#             if len(toks) > max_length:
#                 max_length = len(toks)
#             all.append(len(toks))

In [ ]:
min_length, max_length, np.mean(all), np.median(all), np.std(all)

# Analyse current data and establish a plan for HF upload

In [ ]:
from scripts.data2hf import FEATURES
FEATURES

Image conversion uses `to_uint8` from `scripts/data2hf.py`.

Image and conversation rows use `build_rows` from `scripts/data2hf.py`.

In [ ]:
from scripts.data2hf import upload_data_hf

In [ ]:
sample = out.sample(n=min(500, len(out)), random_state=0)
ds = upload_data_hf(
    sample,
    repo_id="jddomingues/brexpert-mm",
    split="train",
    commit_message="smoke test",
    private=True,
    num_proc=1,
    dry_run=True,
)

d = ds
c = d[0]["conversation"]
assert isinstance(c, list) and isinstance(c[0], dict), type(c)
print(c[0])
print(d[0]["image"].size, d[0]["image"].mode)

In [ ]:
# Remote repository inspection belongs in a separate, intentional Hub session.

In [ ]:
# Remote repository deletion removed from this walkthrough.

In [ ]:
# Local previews do not clear the user's Hugging Face cache.

In [ ]:
ds = upload_data_hf(sample, repo_id="jddomingues/brexpert-mm", split="train", commit_message="Initial upload of Brexpert-MM train split", token="", private=True, num_proc=1, max_shard_size="500MB", dry_run=True)
ds

In [ ]:
print(f"{ds.info.dataset_size / 1e9:.2f} GB")          # arrow bytes
per_row = ds.info.dataset_size / len(ds)
print(f"projected: {per_row * 291_059 / 1e9:.1f} GB")

In [ ]:
import numpy as np
sizes = np.array([len(b["bytes"]) for b in ds.data.column("image").to_pylist()])
mean, sd = sizes.mean(), sizes.std(ddof=1)
se = sd / np.sqrt(len(sizes))
print(f"{mean/1024:.0f} ± {se/1024:.0f} KB/img  (sd {sd/1024:.0f})")
print(f"projected {mean*291_059/1e9:.1f} GB  ±{1.96*se*291_059/1e9:.1f}")
print(f"p50 {np.percentile(sizes,50)/1024:.0f}  p99 {np.percentile(sizes,99)/1024:.0f} KB")

In [ ]:
sample = sample.assign(nbytes=sizes)
per_ds = sample.groupby("dataset").nbytes.mean()
proj = (out.dataset.value_counts() * per_ds).sum()
print(f"stratified projection: {proj/1e9:.1f} GB")

In [ ]:
print(out.dataset.value_counts(normalize=True))
print(sample.dataset.value_counts(normalize=True))

In [ ]:
# Reuse scripts.data2hf.upload_data_hf; the obsolete per-row uploader was removed.

In [ ]:
out.shape

In [ ]:
ds = upload_data_hf(sample, "jddomingues/brexpert-mm", split="train", commit_message="Local preview", num_proc=1, dry_run=True)

In [ ]:
train_dataset = ds

In [ ]:
train_dataset[0]

In [ ]:
sample = out.iloc[0]["conversation"]
sample

In [ ]:
sample_img = out.iloc[0]["image_path"]
sample_img

In [ ]:
img = np.asarray(train_dataset[0]["image"])

In [ ]:
pil_img = Image.fromarray(img)

In [ ]:
pil_img

In [ ]:
plt.imshow(img, cmap="gray")